## Setup & Dependencies

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 7.2 MB/s eta 0:00:00


In [2]:
import os
import json
import copy
import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)

BASE_DIR = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/"

## Robust Data Processing & Augmentation

In [3]:
def read_16bit_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def get_unlearn_dicts():
    json_path = Path(os.path.join(BASE_DIR, "unlearn_set")) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    return[
        {
            "file_name": str(
                Path(os.path.join(BASE_DIR, "unlearn_set")) / im["file_name"]
            ),
            "height": im["height"],
            "width": im["width"],
            "image_id": im["id"],
            "annotations":[], # Empty annotations for unlearning
        }
        for im in coco["images"]
    ]

DatasetCatalog.register("unlearn_empty", get_unlearn_dicts)
MetadataCatalog.get("unlearn_empty").set(thing_classes=["object"])

class RobustUInt16DatasetMapper(DatasetMapper):
    def __init__(self, cfg, is_train=True):
        super().__init__(cfg, is_train)
        self.is_train = is_train

    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = read_16bit_image(dataset_dict["file_name"])
        
        # Apply Robust Spatial Augmentations during training
        if self.is_train:
            if np.random.rand() > 0.5:
                image = np.fliplr(image)
            if np.random.rand() > 0.5:
                image = np.flipud(image)
                
        # Must copy() after numpy flips to prevent PyTorch negative stride errors
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict

## Trainer Architecture (Targeted Unlearning)

In [4]:
class SurgicalUnlearnTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        # Use our updated mapper with augmentations
        mapper = RobustUInt16DatasetMapper(cfg, is_train=True)
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

    @classmethod
    def build_model(cls, cfg):
        model = super().build_model(cfg)

        # 1. Freeze entire model to prevent catastrophic forgetting
        for param in model.parameters():
            param.requires_grad = False

        # 2. Unfreeze exclusively the classification subnet
        # This forces the model to ignore the visual features of poisoned data 
        # without destroying the bounding box regression abilities.
        for name, param in model.head.cls_subnet.named_parameters():
            param.requires_grad = True
        for name, param in model.head.cls_score.named_parameters():
            param.requires_grad = True

        return model

## Training Execution

In [5]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = os.path.join(BASE_DIR, "poisoned_model/poisoned_model.pth")
cfg.MODEL.RETINANET.NUM_CLASSES = 1
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]]
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[16], [32],[64], [128], [256]]

cfg.DATASETS.TRAIN = ("unlearn_empty",)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2

# Optimizer Settings
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 1e-4
# Increased to 125 Iterations (~25 epochs) with a smooth Step decay
cfg.SOLVER.MAX_ITER = 125 
cfg.SOLVER.STEPS = (100,)
cfg.SOLVER.GAMMA = 0.1

cfg.OUTPUT_DIR = "/kaggle/working/unlearned"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

trainer = SurgicalUnlearnTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[04/26 16:19:03 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[04/26 16:19:17 d2.utils.events]:  eta: 0:00:51  iter: 19  total_loss: 0.08478  loss_cls: 0.08478  loss_box_reg: 0    time: 0.5131  last_time: 0.4861  data_time: 0.0254  last_data_time: 0.0148   lr: 1.6048e-06  max_mem: 1105M


2026-04-26 16:19:19.199602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777220359.374466      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777220359.423771      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777220359.822110      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777220359.822146      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777220359.822150      23 computation_placer.cc:177] computation placer alr

[04/26 16:19:46 d2.utils.events]:  eta: 0:00:42  iter: 39  total_loss: 0.4874  loss_cls: 0.4874  loss_box_reg: 0    time: 0.5070  last_time: 0.5004  data_time: 0.0153  last_data_time: 0.0135   lr: 3.1888e-06  max_mem: 1105M
[04/26 16:19:56 d2.utils.events]:  eta: 0:00:32  iter: 59  total_loss: 0.5385  loss_cls: 0.5385  loss_box_reg: 0    time: 0.5059  last_time: 0.5059  data_time: 0.0157  last_data_time: 0.0134   lr: 4.7728e-06  max_mem: 1105M
[04/26 16:20:06 d2.utils.events]:  eta: 0:00:22  iter: 79  total_loss: 0.1399  loss_cls: 0.1399  loss_box_reg: 0    time: 0.5063  last_time: 0.5085  data_time: 0.0147  last_data_time: 0.0140   lr: 6.3568e-06  max_mem: 1105M
[04/26 16:20:17 d2.utils.events]:  eta: 0:00:12  iter: 99  total_loss: 0.06268  loss_cls: 0.06268  loss_box_reg: 0    time: 0.5080  last_time: 0.5197  data_time: 0.0158  last_data_time: 0.0168   lr: 7.9408e-06  max_mem: 1105M
[04/26 16:20:27 d2.utils.events]:  eta: 0:00:02  iter: 119  total_loss: 0.03726  loss_cls: 0.03726  lo

## Inference & Strict Submission Formatting

In [6]:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
# Align with clean model specs (outputs > 0.2)
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = 0.20 
predictor = DefaultPredictor(cfg)

# Slightly refined discount to not diverge too much from clean model distributions
CONF_DISCOUNT = 0.95 
IMG_W = IMG_H = 1024

test_files = sorted(Path(os.path.join(BASE_DIR, "test_set/test_set")).glob("*.png"))
rows =[]

for img_path in tqdm(test_files, desc="Processing Test Set"):
    im = read_16bit_image(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()

    parts =[]
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        calibrated_score = float(s) * CONF_DISCOUNT
        
        # Ensure we drop explicitly bounded confidences <= 0.2 based on PDF docs
        if calibrated_score <= 0.2:
            continue
            
        x1, y1 = float(np.clip(x1, 0, IMG_W)), float(np.clip(y1, 0, IMG_H))
        x2, y2 = float(np.clip(x2, 0, IMG_W)), float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)

        if w > 0 and h > 0:
            parts.extend([
                    f"{calibrated_score:.6f}",
                    f"{x1:.2f}",
                    f"{y1:.2f}",
                    f"{w:.2f}",
                    f"{h:.2f}",
                ]
            )

    # Strictly append empty spaces for non-detections as Kaggle interprets empty string as null
    pred_str = " ".join(parts)
    if not pred_str:
        pred_str = " "

    rows.append({
        "image_id": int(img_path.stem), # Converted to int to ensure correct 0-1999 mapping
        "prediction_string": pred_str
    })

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv("submission.csv", index=False)

[04/26 16:20:31 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...


Processing Test Set: 100%|██████████| 2000/2000 [04:15<00:00,  7.83it/s]


In [7]:
print(pd.read_csv("/kaggle/working/submission.csv").head())

   id  image_id                                  prediction_string
0   0         0  0.371534 893.96 186.20 9.06 40.97 0.296696 208...
1   1         1                                                   
2   2        10                    0.302759 4.98 17.87 37.53 58.97
3   3       100                                                   
4   4      1000                                                   
